# 3: Explore topics and their supporting passages

An entity graph asks how particular things are connected. A topic groups related passages so you can explore a broader area of the corpus. Keep this follow-on short: inspect the captured hierarchy, open its evidence, and compare direct passage neighbors with topic-routed neighbors.

> **Sidebar:** Follow [the topics checkpoints](../../../instructions/lab-notebooks.md#notebook-3-explore-topics-without-confusing-them-with-entities). Use the same prepared Search service and graph-lab kernel. This notebook runs in a fresh kernel and only reads the indexes restored by the existing graph loader.

No topics are generated, named, or silently repaired here. Unnamed groups and partial coverage remain visible. Query vectors come from a captured source passage, so no embedding model is called. This is **not** the demo's native `topicGraph` diversification modifier: the loader restores sidecars but does not link that modifier to a vector field.

## Step 1: Locate the restored topic indexes

Read physical names from the committed graph inventory, then apply the same loader prefix used in the entity notebook. All connections use `AZURE_SEARCH_SERVICE_ENDPOINT` and its credential; there is no second service or private demo profile.

In [ ]:
import json
import math
from pathlib import Path
import sys
from time import perf_counter

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.env.sample').is_file()), None)
if root is None:
    raise RuntimeError('Open this notebook from inside the LAB673 repository.')
sys.path.insert(0, str(root / 'src' / 'notebooks' / 'lab'))
from lab_helpers import load_settings, require_complete, show_json, show_table, show_topic_hierarchy
from azure.search.documents.models import VectorizedQuery

settings = load_settings()
inventory = json.loads((settings.root / 'data' / 'graph-data' / 'manifest.json').read_text(encoding='utf-8'))
logical_names = {entry['definition']: entry['index'] for entry in inventory['restore']['order']}
TOPICS_INDEX = settings.graph_index(logical_names['cocoarynth-topics-index.json'])
TOPIC_VECTORS_INDEX = settings.graph_index(logical_names['cocoarynth-topic-vectors-index.json'])
SOURCE_INDEX = settings.graph_index(logical_names['cocoarynth-topic-sources-index.json'])
with settings.index_client() as indexes:
    for name in (TOPICS_INDEX, TOPIC_VECTORS_INDEX, SOURCE_INDEX):
        indexes.get_index(name)
print('Topic representatives, vectors, and original source chunks are on the configured service.')

## Step 2: Read coverage before the picture

The sidecar manifest records its generation, source binding, partial coverage, and vector provenance. Its `source_document_count` describes the snapshot's source chunks, not the number of original PDFs.

The captured build assigned 97 of 106 source chunks; nine were unassigned. It recorded six native topic vectors and two missing parent vectors. The separate restored topic-vector index also includes two **derived parent centroids**. Do not call those native vectors, infer missing labels, or present the hierarchy as complete corpus coverage.

In [ ]:
with settings.search_client(TOPICS_INDEX) as topics:
    manifest_rows = list(topics.search(
        search_text='*', filter="kind eq 'manifest'", select=['payload'], top=2,
        raw_response_hook=require_complete,
    ))
if len(manifest_rows) != 1:
    raise RuntimeError('Expected exactly one captured topic manifest.')
topic_manifest = json.loads(manifest_rows[0]['payload'])
if topic_manifest['physical_index'] != TOPICS_INDEX or topic_manifest['source_index'] != SOURCE_INDEX:
    raise RuntimeError('The restored topic manifest does not bind to these physical indexes.')
GENERATION = topic_manifest['generation_id']
show_json({key: topic_manifest[key] for key in (
    'generation_id', 'source_document_count', 'assigned_node_count',
    'unassigned_source_document_count', 'node_coverage', 'vector_coverage',
    'label_stage', 'labels_from_native_binary',
)})

## Step 3: Browse the actual hierarchy

Representatives connect through stored `parent_id` values. A parent-child topic link is not a business relationship such as a farm supplying a lot. Membership counts can overlap across levels; do not add them as unique document counts.

The diagram uses stored labels when present and community ID prefixes otherwise. The table keeps the full IDs, levels, label status, and centroid method visible.

In [ ]:
def odata_string(value):
    return "'" + value.replace("'", "''") + "'"

generation_filter = 'generation_id eq ' + odata_string(GENERATION)
with settings.search_client(TOPICS_INDEX) as topics:
    representatives = list(topics.search(
        search_text='*', filter="kind eq 'representative' and " + generation_filter,
        select=['community_id', 'parent_id', 'level', 'member_count', 'label', 'label_status', 'centroid_method'],
        top=1000, raw_response_hook=require_complete,
    ))
show_topic_hierarchy(representatives)
show_table(sorted(representatives, key=lambda row: (-row['level'], row['community_id'])),
           ['community_id', 'parent_id', 'level', 'member_count', 'label', 'label_status', 'centroid_method'],
           'Captured topic representatives; unnamed groups stay unnamed')

## Step 4: Choose an original passage as the exploration seed

Find a passage mentioning Madagascar lot MG26-L02, read it, and reuse its stored embedding. This asks **what is near this passage**, not what a new natural-language question would retrieve after embedding. Both comparisons below use exactly that same vector and may include the seed itself.

The original source text is evidence. A centroid is a vector representation of a group, not a quotation or a generated summary.

In [ ]:
with settings.search_client(SOURCE_INDEX) as sources:
    seeds = list(sources.search(
        search_text='"MG26-L02"', search_fields=['chunk'],
        select=['chunk_id', 'title', 'blob_path', 'chunk', 'text_vector'], top=1,
        raw_response_hook=require_complete,
    ))
if not seeds:
    raise RuntimeError('The captured source passage is missing. Check the topic-source restoration.')
seed = seeds[0]
query_vector = seed['text_vector']
if len(query_vector) != 3072 or not all(isinstance(value, (int, float)) and math.isfinite(value) for value in query_vector):
    raise RuntimeError('The seed needs its captured 3072-dimensional embedding.')
show_table([seed], ['chunk_id', 'title', 'blob_path', 'chunk'], 'Read the actual seed passage; the vector is not displayed')

## Step 5: Retrieve nearest passages directly

This SDK vector query searches the captured source vectors directly. Preserve the service's returned order. The score is a retrieval score, not a probability that a passage is correct.

In [ ]:
PASSAGE_FIELDS = ['chunk_id', 'title', 'blob_path', 'chunk']
with settings.search_client(SOURCE_INDEX) as sources:
    started = perf_counter()
    direct = list(sources.search(
        vector_queries=[VectorizedQuery(vector=query_vector, fields='text_vector', k_nearest_neighbors=8, exhaustive=True)],
        select=PASSAGE_FIELDS, top=8, raw_response_hook=require_complete,
    ))
    direct_ms = (perf_counter() - started) * 1000
if not direct:
    raise RuntimeError('Direct vector retrieval returned no captured passages.')
show_table([{'rank': rank, **row} for rank, row in enumerate(direct, 1)],
           ['rank', 'title', 'blob_path', 'chunk'], 'Direct passage neighbors in service order')

## Step 6: Find a native topic, then rank its member passages

First find the nearest **native level-0 centroid**, excluding derived parent centroids. Then look up its captured members and rank only those source chunks with the same seed vector. The topic narrows the candidate set; it may omit relevant material in another group or among the unassigned chunks.

This explicit SDK routing is separate from the service's native `topicGraph` query modifier. It exposes the representative-to-members-to-passages mechanism without changing the loader's index definitions or pretending the captured sidecar is a complete taxonomy.

In [ ]:
started = perf_counter()
with settings.search_client(TOPIC_VECTORS_INDEX) as vectors:
    nearest = list(vectors.search(
        vector_queries=[VectorizedQuery(vector=query_vector, fields='centroid', k_nearest_neighbors=1, exhaustive=True)],
        filter="centroid_method eq 'native-terminal-representative-normalized-float32' and level eq 0 and " + generation_filter,
        select=['community_id', 'generation_id', 'centroid_method', 'member_count'], top=1,
        raw_response_hook=require_complete,
    ))
if not nearest:
    raise RuntimeError('No native level-0 centroid was returned; do not substitute a derived parent.')
chosen = nearest[0]
with settings.search_client(TOPICS_INDEX) as topics:
    memberships = list(topics.search(
        search_text='*',
        filter="kind eq 'membership' and community_id eq " + odata_string(chosen['community_id']) + ' and ' + generation_filter,
        select=['chunk_id'], top=1000, raw_response_hook=require_complete,
    ))
member_ids = sorted({row['chunk_id'] for row in memberships})
if not member_ids or any(',' in value for value in member_ids):
    raise RuntimeError('Expected captured membership IDs compatible with this bounded filter.')
member_filter = 'search.in(chunk_id, ' + odata_string(','.join(member_ids)) + ", ',')"
with settings.search_client(SOURCE_INDEX) as sources:
    routed = list(sources.search(
        vector_queries=[VectorizedQuery(vector=query_vector, fields='text_vector', k_nearest_neighbors=8, exhaustive=True)],
        filter=member_filter, select=PASSAGE_FIELDS, top=8, raw_response_hook=require_complete,
    ))
routed_ms = (perf_counter() - started) * 1000
if not routed or any(row['chunk_id'] not in member_ids for row in routed):
    raise RuntimeError('Topic-routed results did not match the captured membership scope.')
show_table([chosen], ['community_id', 'centroid_method', 'member_count'], 'The actual selected native level-0 topic')
print(f'Captured members: {len(member_ids)}; seed belongs to this group: {seed["chunk_id"] in member_ids}')
show_table([{'rank': rank, **row} for rank, row in enumerate(routed, 1)],
           ['rank', 'title', 'blob_path', 'chunk'], 'Nearest passages within that captured topic, in service order')

## Step 7: Inspect differences without declaring a winner

The direct path performs one source-vector search. The routed path also performs centroid and membership lookups. The times below describe these observed read paths and exclude new query embedding, topic construction, and answer generation. They are not end-to-end question latency or a quality benchmark.

A passage appearing only on one side means it is absent from the other's returned list, not absent from the index. Scores from centroid lookup and passage lookup are not calibrated for comparison.

In [ ]:
direct_ids = {row['chunk_id'] for row in direct}
routed_ids = {row['chunk_id'] for row in routed}
show_table([
    {'path': 'Direct passage neighbors', 'returned': len(direct), 'client ms': round(direct_ms), 'candidate scope': 'All captured source chunks'},
    {'path': 'Native centroid then captured members', 'returned': len(routed), 'client ms': round(routed_ms), 'candidate scope': f'{len(member_ids)} captured members'},
], ['path', 'returned', 'client ms', 'candidate scope'], 'Observed passage-seeded exploration, not a full query benchmark')
show_table([{
    'shared returned chunks': len(direct_ids & routed_ids),
    'direct-list only': len(direct_ids - routed_ids),
    'topic-list only': len(routed_ids - direct_ids),
}], ['shared returned chunks', 'direct-list only', 'topic-list only'], 'Returned-list overlap does not establish relevance or topic coverage')

## Choose the representation for the question

| Need | Useful starting point | Tradeoff to inspect |
| --- | --- | --- |
| Ground a direct question in original passages | Standard knowledge-base retrieval | Planning, source selection, citations, and synthesis work |
| Respond quickly within a known narrow source scope | MR1 | Faster planning is the objective; the application supplies source scope and pure MR1 here returns evidence, not an answer |
| Follow explicit entity relationships across records | Entity graph and GQL | Inspectable joins and provenance, with extraction/identity limits and potentially more model reasoning |
| Explore groups of related passages | Topic representatives and membership evidence | Broader organization, but unnamed groups, partial coverage, and candidate-scope tradeoffs remain |

**Sidebar checkpoint:** Explain why the organic-seal question, the batch-to-farm question, and this topic browse call for different evidence. More structure does not make one approach universally better.

The shared indexes and Graph KS remain unchanged. Clear all notebook outputs before saving.